### In the AIDev dataset, Human PRs only exist for repos ≥500 stars. The AIDev-pop only has agent data for repos ≥500 stars. We started with the Human data, and then combined the same repositories that existed from all_repository

In [ ]:
!pip install polars pyarrow --quiet
import polars as pl

# URLs
human_pr_url = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/human_pull_request.parquet"
all_pr_url    = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/all_pull_request.parquet"
repo_url      = "https://huggingface.co/datasets/hao-li/AIDev/resolve/main/all_repository.parquet"

# ----------------------------
# 1. Load data
# ----------------------------
print("Loading datasets...")
human_pr = pl.read_parquet(human_pr_url)
all_pr   = pl.read_parquet(all_pr_url)
repos    = pl.read_parquet(repo_url)

print("Loaded shapes:")
print("Human PRs:", human_pr.shape)
print("All PRs:", all_pr.shape)
print("Repositories:", repos.shape)

# ----------------------------
# 2. Identify repos used in Human PR dataset
# ----------------------------
human_repo_ids = human_pr.select("repo_url").unique()

print("\nUnique human repo URLs:", human_repo_ids.height)

# Some datasets use `repo_id` instead of URL — OPTIONAL: extract repo_id as well
if "repo_id" in human_pr.columns:
    human_repo_ids = human_pr.select("repo_id").unique()

# ----------------------------
# 3. Filter agent PRs to those repos
# ----------------------------
print("\nFiltering agent PRs to human PR repos...")

# Filter agent PRs: agent != "Human"
agent_pr = all_pr.filter(pl.col("agent") != "Human")

# Join using repo_url if available
if "repo_url" in agent_pr.columns and "repo_url" in human_pr.columns:
    agent_pr_matched = agent_pr.join(
        human_pr.select("repo_url").unique(),
        on="repo_url",
        how="inner"
    )
else:
    # Fallback: join on repo_id
    agent_pr_matched = agent_pr.join(
        human_pr.select("repo_id").unique(),
        on="repo_id",
        how="inner"
    )

print("Filtered agent PRs:", agent_pr_matched.shape)

# ----------------------------
# 4. Align schemas before concat
# ----------------------------

# find missing columns
human_cols = set(human_pr.columns)
agent_cols = set(agent_pr_matched.columns)

missing_in_human = agent_cols - human_cols
missing_in_agent = human_cols - agent_cols

print("Missing in human:", missing_in_human)
print("Missing in agent:", missing_in_agent)

# Add missing columns to human_pr (with null)
for col in missing_in_human:
    human_pr = human_pr.with_columns(pl.lit(None).alias(col))

# Add missing columns to agent_pr_matched (with null)
for col in missing_in_agent:
    agent_pr_matched = agent_pr_matched.with_columns(pl.lit(None).alias(col))

# Reorder columns to match
agent_pr_matched = agent_pr_matched.select(sorted(agent_pr_matched.columns))
human_pr = human_pr.select(sorted(human_pr.columns))

# ----------------------------
# 5. Concat
# ----------------------------
final_dataset = (
    pl.concat([human_pr, agent_pr_matched], how="vertical_relaxed")
    .with_columns(
        (pl.col("agent") == "Human").alias("is_human")
    )
)

print("\nFinal dataset:", final_dataset.shape)

# ----------------------------
# 6. Save file
# ----------------------------
output_path = "/content/human_agent_same_repos.parquet"
final_dataset.write_parquet(output_path)

print(f"\nSaved unified dataset to: {output_path}")



Uploaded this to https://huggingface.co/datasets/kaylamarietorres/AIDev/human_agent_same_repos.parquet